In [26]:
from matching_clustering.utils import LapOT
from matching_clustering.create_clouds import create_clouds, plot_point_clouds_side_by_side
from matching_clustering.graph_laplacians_and_c import graph_laplacians,cost_matrix_c
from matching_clustering.refined_simultaneous_clustering import build_cluster_matrix, spectral_clustering_from_W, plot_clouds_side_by_side_3d


#Set and visualize the clouds

In [27]:
path_X = "Data/CAPOD/class1/m1.obj"
path_Y = "Data/CAPOD/class1/m7.obj"
X, Y = create_clouds(path_X, path_Y)
plot_point_clouds_side_by_side(X, Y)


In [28]:
sigma_clustering = 0.1
L_X, L_Y, W_x, W_y, a, b = graph_laplacians(X, Y, sigma_x=sigma_clustering, sigma_y=sigma_clustering)

W1_matrix = cost_matrix_c(X, Y, a, b)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.1s
[Parallel(n_jobs=-1)]: Done 1420 tasks      | elapsed:    2.3s
[Parallel(n_jobs=-1)]: Done 56844 tasks      | elapsed:    6.2s
[Parallel(n_jobs=-1)]: Done 146444 tasks      | elapsed:   12.6s
[Parallel(n_jobs=-1)]: Done 261644 tasks      | elapsed:   20.6s
[Parallel(n_jobs=-1)]: Done 402444 tasks      | elapsed:   30.2s
[Parallel(n_jobs=-1)]: Done 568844 tasks      | elapsed:   41.5s
[Parallel(n_jobs=-1)]: Done 760844 tasks      | elapsed:   54.7s
[Parallel(n_jobs=-1)]: Done 978444 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 1221644 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 1490444 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 1784844 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done 2104844 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 2450444 tasks      | elapsed:  2.9min
[Parallel(n_jo

In [29]:
# Cost matrix
C = W1_matrix

# Laplacian matrices
K = L_X
L = L_Y

# Set hyperparameters
l = 0.7  # lambda
l_x = 1
l_y = 1

# optimize our objective
prob = LapOT(K, L, C, l, l_x, l_y, w=a, v=b)
pi_fp = prob.solve(method='fp', verbose=True)

# derive the switch matrices
pi_switch_x = build_cluster_matrix(pi_fp, k=2)
pi_switch_y = build_cluster_matrix(pi_fp.T, k=2)

# refine the similarity matrices
new_similarity_x, new_similarity_y = pi_switch_x*W_x, pi_switch_y*W_y

# Run spectral clustering based on the refined similarity matrices
labels_X, L_X, U_X, k_X = spectral_clustering_from_W(new_similarity_x, n_clusters=5, random_state=1)
labels_Y, L_Y, U_Y, k_Y = spectral_clustering_from_W(new_similarity_y, n_clusters=5, random_state=1)

print(f"[Info] Chosen k for cloud X: {k_X}")
print(f"[Info] Chosen k for cloud Y: {k_Y}")

# Visualize the clusters side by side
fig = plot_clouds_side_by_side_3d(
    X, labels_X, Y, labels_Y,
    title_left=f"Cloud X (k={k_X})", title_right=f"Cloud Y (k={k_Y})"
)
fig.show()

Initialization: objective = -10.77329640760788
Implement fixed point algorithm
max of entropy term :-10.991969981637103 
 max of main term: 0.21387562356376158
Iteration 1: objective = -10.778094358073341, objective relative change = -0.0004453558394692203, pi change (L1) = 0.09192787368871717
max of entropy term :-10.991970234480071 
 max of main term: 0.21387587636684574
Iteration 2: objective = -10.778094358113226, objective relative change = -3.700518736429346e-12, pi change (L1) = 7.685405835094268e-06
Terminated after 2 iterations (coupling converged)
[Info] Chosen k for cloud X: 5
[Info] Chosen k for cloud Y: 5
